# 09. Production Inference & Customer Risk Classification

## Executive Summary
This notebook demonstrates real-world inference on new customer records. It covers:
1. Loading the production pipeline (`preprocessor.pkl` and `best_model.pkl`).
2. Single customer prediction with churn probability calculation.
3. Multi-tier Risk Classification:
   - **Low Risk (< 0.30):** Loyal customer. Maintain standard experience.
   - **Medium Risk (0.30 - 0.60):** Warning signs. Survey, review tech support, offer annual discount.
   - **High Risk (>= 0.60):** Critical danger. Immediate proactive retention specialist outreach.
4. Batch prediction across customer segments.
5. Integration with the reusable Python prediction module (`src.prediction`).

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

pd.set_option("display.max_columns", None)

preprocessor = joblib.load("../models/preprocessor.pkl")
model = joblib.load("../models/best_model.pkl")

print("Preprocessor and Best Model loaded successfully.")

Preprocessor and Best Model loaded successfully.


## 1. Customer Profiles for Inference

In [2]:
# Define 3 representative customer profiles
customer_high_risk = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 3,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 92.50,
    "TotalCharges": 277.50,
}

customer_loyal = {
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "Yes",
    "tenure": 60,
    "PhoneService": "Yes",
    "MultipleLines": "Yes",
    "InternetService": "DSL",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "Yes",
    "DeviceProtection": "Yes",
    "TechSupport": "Yes",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Two year",
    "PaperlessBilling": "No",
    "PaymentMethod": "Credit card (automatic)",
    "MonthlyCharges": 48.00,
    "TotalCharges": 2880.00,
}

customer_new = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 0,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "DSL",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "No",
    "PaymentMethod": "Mailed check",
    "MonthlyCharges": 24.50,
    "TotalCharges": " ",  # Testing edge case blank string
}

## 2. Reusable Prediction Function with Risk Tiers

In [3]:
def predict_customer(customer_dict, decision_threshold=0.50):
    df_raw = pd.DataFrame([customer_dict])
    
    # Clean whitespace in TotalCharges if present
    if df_raw["TotalCharges"].dtype == object:
        df_raw["TotalCharges"] = pd.to_numeric(df_raw["TotalCharges"].astype(str).str.strip(), errors="coerce").fillna(0.0)
    
    # Preprocess
    X_proc = preprocessor.transform(df_raw)
    feature_names = preprocessor.get_feature_names_out()
    X_proc_df = pd.DataFrame(X_proc, columns=feature_names)
    
    # Probabilities
    classes = list(getattr(model, "classes_", [0, 1]))
    pos_idx = classes.index(1) if 1 in classes else (classes.index("Yes") if "Yes" in classes else 1)
    prob = model.predict_proba(X_proc_df)[0, pos_idx]
    
    prediction = "Churn" if prob >= decision_threshold else "No Churn"
    
    if prob < 0.30:
        risk = "Low Risk"
        action = "Customer shows strong loyalty. Maintain standard experience; consider cross-selling."
    elif prob < 0.60:
        risk = "Medium Risk"
        action = "Customer exhibits moderate risk. Conduct service check-in, offer 1-year contract incentive."
    else:
        risk = "High Risk"
        action = "Critical churn risk! Urgent retention outreach; offer discounted long-term bundle."
        
    return {
        "Prediction": prediction,
        "Churn Probability": f"{prob * 100:.2f}%",
        "Risk Tier": risk,
        "Retention Action": action,
    }

## 3. Testing Single Customer Predictions

In [4]:
print("--- Profile 1: Short-Tenure Month-to-Month Customer ---")
res1 = predict_customer(customer_high_risk)
for k, v in res1.items():
    print(f"  {k:20s}: {v}")

print("\n--- Profile 2: Long-Tenure Two-Year Contract Customer ---")
res2 = predict_customer(customer_loyal)
for k, v in res2.items():
    print(f"  {k:20s}: {v}")

print("\n--- Profile 3: Brand New Customer (tenure=0, blank charges) ---")
res3 = predict_customer(customer_new)
for k, v in res3.items():
    print(f"  {k:20s}: {v}")

--- Profile 1: Short-Tenure Month-to-Month Customer ---
  Prediction          : Churn
  Churn Probability   : 76.67%
  Risk Tier           : High Risk
  Retention Action    : Critical churn risk! Urgent retention outreach; offer discounted long-term bundle.

--- Profile 2: Long-Tenure Two-Year Contract Customer ---
  Prediction          : No Churn
  Churn Probability   : 0.72%
  Risk Tier           : Low Risk
  Retention Action    : Customer shows strong loyalty. Maintain standard experience; consider cross-selling.

--- Profile 3: Brand New Customer (tenure=0, blank charges) ---
  Prediction          : No Churn
  Churn Probability   : 45.44%
  Risk Tier           : Medium Risk
  Retention Action    : Customer exhibits moderate risk. Conduct service check-in, offer 1-year contract incentive.


## 4. Batch Prediction Demonstration

In [5]:
batch_customers = [customer_high_risk, customer_loyal, customer_new]
batch_results = [predict_customer(c) for c in batch_customers]
batch_df = pd.DataFrame(batch_results, index=["Customer A (High Risk)", "Customer B (Loyal)", "Customer C (New)"])
display(batch_df)

,Prediction,Churn Probability,Risk Tier,Retention Action
Customer A (High Risk),Churn,76.67%,High Risk,Critical churn risk! Urgent retention outreach...
Customer B (Loyal),No Churn,0.72%,Low Risk,Customer shows strong loyalty. Maintain standa...
Customer C (New),No Churn,45.44%,Medium Risk,Customer exhibits moderate risk. Conduct servi...


## 5. Using the Production Python Package (`src.prediction`)

In [6]:
import sys
sys.path.append("..")
from src.prediction.predict import predict_churn

result = predict_churn(customer_high_risk)
print("Result from src.prediction module:")
print("  Prediction        :", result.prediction)
print("  Churn Probability :", f"{result.churn_probability * 100:.2f}%")
print("  Risk Level        :", result.risk_level)
print("  Recommendation    :", result.recommendation)

Result from src.prediction module:
  Prediction        : Churn
  Churn Probability : 76.67%
  Risk Level        : High Risk
  Recommendation    : Immediate proactive retention required! Assign retention specialist, offer long-term contract pricing incentive, and review recent technical support tickets.
